# Chapter 9: Failure Detection
*From: Database Internals*

## TL;DR

- A **failure detector** is a local subsystem that marks unresponsive peers as "suspected" so the rest of the system can make progress; it is a prerequisite for consensus and atomic broadcast.
- In an asynchronous network, you cannot distinguish a crashed process from a slow one — every detector trades **accuracy** against **efficiency** and tolerates some rate of false positives / false negatives.
- Key algorithmic properties: **completeness** (every real failure is eventually seen by every nonfaulty node), **efficiency** (how fast), **accuracy** (how few wrong suspicions).
- Five families of algorithms in this chapter: **timeout-free counter propagation**, **SWIM outsourced heartbeats**, **phi-accrual** (continuous suspicion), **gossip** heartbeat tables, and **FUSE** (quiescence as propagation).
- All of them assume non-Byzantine behavior (nodes may crash or be slow, but do not lie).

## Requirements

### Functional Requirements

- Every nonfaulty process must eventually detect the failure of every faulty process (**completeness**).
- Surface suspicion to the consuming algorithm (consensus, replication, leader election) with a stable API — either a boolean "alive/suspect" flag or a continuous suspicion score.
- Tolerate **link-level** failures (lost or delayed messages) as well as **process-level** crashes.
- Operate without synchronized clocks or fixed message-delay bounds (asynchronous model).

### Non-Functional Requirements

- **Bandwidth**: message rate should not explode with cluster size — prefer sub-linear or at worst linear growth in N.
- **Tuning**: thresholds (timeout, phi, counter gap) should be either adaptive or insensitive to normal jitter.
- **Partition tolerance**: a single faulty link between two peers should not cause spurious suspicions if another path exists.
- **Latency of detection**: should be bounded by a small multiple of the heartbeat interval, not minutes.
- **No Byzantine trust assumptions needed** — all discussed algorithms assume honest participants.

## Back-of-Envelope Estimation

Estimate heartbeat traffic, message rate, and per-approach memory so you can pick a detector that fits the cluster.

In [ ]:
# Failure-detection capacity estimation
# Scenario: cluster of N nodes, heartbeat interval T seconds, small msg size.

N = 1_000                      # cluster size
T = 1.0                        # heartbeat interval (seconds)
msg_bytes = 200                # heartbeat payload (counter + small metadata)

# --- 1. All-to-all heartbeats (naive: every node pings every other node) ---
all_to_all_msgs_per_sec = N * (N - 1) / T
all_to_all_bw_mbps = all_to_all_msgs_per_sec * msg_bytes * 8 / 1e6
print(f"All-to-all:       {all_to_all_msgs_per_sec:>12,.0f} msg/s   {all_to_all_bw_mbps:7.2f} Mbps")

# --- 2. Gossip (each node pushes its table to 1 random peer per round) ---
# Table carries N (id, counter, ts) ~= 40 bytes each.
gossip_payload = N * 40
gossip_msgs_per_sec = N / T
gossip_bw_mbps = gossip_msgs_per_sec * gossip_payload * 8 / 1e6
print(f"Gossip:           {gossip_msgs_per_sec:>12,.0f} msg/s   {gossip_bw_mbps:7.2f} Mbps")

# --- 3. SWIM (each node probes 1 random peer per round; on miss, k indirect probes) ---
k = 3                          # indirect-probe fan-out
miss_rate = 0.02               # 2% of direct probes miss and trigger indirect
swim_msgs_per_sec = N / T + N * miss_rate * k / T
swim_bw_mbps = swim_msgs_per_sec * msg_bytes * 8 / 1e6
print(f"SWIM:             {swim_msgs_per_sec:>12,.0f} msg/s   {swim_bw_mbps:7.2f} Mbps")

# --- 4. Phi-accrual sliding-window memory per monitored peer ---
window = 1000                  # samples per peer
bytes_per_sample = 8           # int64 arrival timestamp
phi_mem_per_peer = window * bytes_per_sample
phi_mem_total_mb = N * phi_mem_per_peer / 1e6
print(f"Phi-accrual mem:  {phi_mem_per_peer} B/peer   {phi_mem_total_mb:7.2f} MB total (all peers monitored)")

# --- 5. Timeout-free: path grows until all N ids are present; worst-case payload ---
id_bytes = 8
tf_worst_payload = N * id_bytes
print(f"Timeout-free worst-case heartbeat payload: {tf_worst_payload} B ({tf_worst_payload/1024:.1f} KB)")

**Reading the numbers.** All-to-all becomes untenable well before N = 1000 (scales as N^2). Gossip grows linearly but the *payload* also grows with N, so effective bandwidth is O(N^2) bytes/sec — still much cheaper in messages. SWIM keeps message count roughly linear and payload small. Phi-accrual memory is per-peer, so monitoring a full 1000-node cluster from one vantage point costs ~8 MB of sample buffers.

## High-Level Design

Every failure detector shares the same three-stage pipeline; only the mechanics differ.

```mermaid
graph LR
  subgraph Peer["Each Process"]
    MON[Monitoring<br/>pings / heartbeats / samples] --> INT[Interpretation<br/>timeout / phi / counter / quiescence]
    INT --> ACT[Action<br/>emit suspect event]
  end
  ACT --> CONS[Consumer:<br/>consensus, replication,<br/>leader election, membership]
```

The **monitoring** stage collects liveness evidence. The **interpretation** stage turns that evidence into a verdict. The **action** stage invokes a callback — usually "remove this peer from the quorum" — that a higher-level algorithm acts on.

## Deep Dive 1: Timeout-Free Failure Detector

Aguilera's timeout-free algorithm removes clocks from the picture entirely. Each heartbeat carries the **path** it has traveled; receivers **increment counters** for every id in that path and re-forward along un-visited edges. Suspicion is a function of counter gaps, not wall time.

```mermaid
sequenceDiagram
  participant P1
  participant P2
  participant P3
  participant P4
  P1->>P2: HB path=[P1]
  P2->>P3: HB path=[P1,P2]  (P2 increments P1's counter)
  P2->>P4: HB path=[P1,P2]
  P3->>P4: HB path=[P1,P2,P3] (P3 increments P1,P2)
  Note over P4: Stops forwarding: all IDs in path.
  Note over P1,P4: Link P1<->P3 is faulty, yet P3 still sees P1's counter rise via P2.
```

**Why it works.** Counters are a *global, normalized* view: every correct process eventually sees every other process's counter tick upward, even across faulty direct links, as long as a **fair path** exists. No timeouts means no timing assumptions — the algorithm runs under the true asynchronous model.

**Why it is hard to use.** You still have to pick a **counter-gap threshold** that distinguishes "this peer is slow but alive" from "this peer is dead." The algorithm gives you sound signal but does not tell you where to put the cut.

## Deep Dive 2: SWIM Outsourced Heartbeats

SWIM (Scalable Weakly-consistent Infection-style Process-group Membership) refuses to trust a single vantage point. When P1 does not hear back from P2, it asks **k random peers** to probe P2 on its behalf.

```mermaid
sequenceDiagram
  participant P1
  participant P2
  participant P3
  participant P4
  P1->>P2: PING
  Note over P1: Timeout: no ack.
  P1->>P3: REQ(ping P2)
  P1->>P4: REQ(ping P2)
  P3->>P2: PING
  P4->>P2: PING
  P2-->>P3: ACK
  P2-->>P4: ACK
  P3-->>P1: ACK(P2 alive)
  P4-->>P1: ACK(P2 alive)
  Note over P1: At least one indirect ack -> keep P2 alive.
```

**Key ideas.**
- Each node tracks only a **subset** of peers, not the whole membership — bandwidth stays linear.
- Indirect probes eliminate false positives caused by a single failing link between P1 and P2.
- Because indirect probes fan out **in parallel**, detection latency barely grows.

SWIM is the membership core of HashiCorp Serf, Consul, and Cassandra's gossiper (blended with phi).

## Deep Dive 3: Phi-Accrual Failure Detector

Hayashibara's phi-accrual detector stops treating liveness as a Boolean. It maintains a **sliding window** of recent heartbeat inter-arrival times, fits a normal distribution (mean mu, variance sigma^2), and emits a continuous **suspicion level phi** proportional to `-log10(P(next heartbeat arrives this late | distribution))`.

```mermaid
graph LR
  HB[Heartbeat<br/>arrivals] --> W[Sliding Window<br/>last W samples]
  W --> S[Statistics<br/>mu, sigma]
  S --> P[phi = -log10 P_later]
  P -->|phi >= threshold| SUSP[Mark suspect]
  P -->|phi < threshold| OK[Keep alive]

  subgraph Monitoring
    HB
    W
  end
  subgraph Interpretation
    S
    P
  end
  subgraph Action
    SUSP
    OK
  end
```

**Why phi is nicer than a hard timeout.**
- The distribution is **learned** from the actual network, so a jittery WAN link and a quiet LAN both get calibrated thresholds.
- Downstream code can consume phi directly and make **graded decisions** — slow down writes at phi=3, exclude from quorum at phi=8, evict at phi=12.
- When the network degrades globally, mu and sigma grow, and phi rises more slowly — the detector becomes less trigger-happy exactly when you want it to.

**Knobs that still matter.** Window size W (too small = noisy, too large = slow to react), the phi threshold, and what you do with a suspect (the *Action* callback) are all workload-specific. Cassandra and Akka both ship phi-accrual with defaults that work for most clusters but can be tuned per deployment.

## Deep Dive 4: Gossip-Based Failure Detection

van Renesse's gossip detector keeps, on every node, a **table** of `(peer_id, heartbeat_counter, local_timestamp)`. Every period each node (a) bumps its own counter and (b) sends the whole table to one random peer. Receivers merge by taking the max counter. A peer whose counter has not advanced in the receiver's view for longer than a threshold is declared dead.

```mermaid
graph TB
  subgraph A["(a) Healthy"]
    P1a[P1<br/>t1=5 t2=5 t3=5] <--> P2a[P2<br/>t1=5 t2=5 t3=5]
    P2a <--> P3a[P3<br/>t1=5 t2=5 t3=5]
    P1a <--> P3a
  end
  subgraph B["(b) Link P1-P3 broken"]
    P1b[P1<br/>t1=6 t2=6 t3=6] <--> P2b[P2<br/>t1=6 t2=6 t3=6]
    P2b <--> P3b[P3<br/>t1=6 t2=6 t3=6]
    P1b -.X.-> P3b
  end
  subgraph C["(c) P3 crashed"]
    P1c[P1<br/>t1=8 t2=8 t3=6] <--> P2c[P2<br/>t1=8 t2=8 t3=6]
    P2c -.X.- P3c[P3<br/>CRASHED]
    P1c -.X.- P3c
  end
```

**Properties.**
- Worst-case bandwidth is **linear** in N per node per round (one table exchange).
- A broken link between two peers is masked as long as *any* gossip path connects them — partition-robust.
- Merging by max counter makes the detector eventually consistent across the cluster: every node converges to the same view.
- The timeout for "counter hasn't advanced" must be picked carefully — too tight produces false positives during gossip round lag.

## Deep Dive 5: FUSE — Reversing Failure Detection

Dunagan's FUSE turns the problem on its head: instead of reliably broadcasting "P2 is dead," it uses **quiescence** — *stopping* responses — as the propagation channel. One failure becomes a group failure.

```mermaid
graph LR
  subgraph S1["(a) Healthy"]
    A1[P1] --- A2[P2]
    A1 --- A3[P3]
    A1 --- A4[P4]
    A2 --- A3
    A2 --- A4
    A3 --- A4
  end
  subgraph S2["(b) P2 crashes"]
    B1[P1] --- B3[P3]
    B1 --- B4[P4]
    B3 --- B4
    B2[P2<br/>X]
  end
  subgraph S3["(c) P4 sees it, goes silent"]
    C1[P1] --- C3[P3]
    C4[P4<br/>quiet]
    C3 -.X.- C4
    C1 -.X.- C4
  end
  subgraph S4["(d) Silence propagates -> group failure"]
    D1[P1<br/>quiet]
    D3[P3<br/>quiet]
    D4[P4<br/>quiet]
    D2[P2<br/>X]
  end
```

**Why quiescence.** You don't have to *send* anything to propagate the news — an absent ping reply *is* the news. That makes FUSE robust to partitions: a partition cannot stop silence from spreading.

**The cost.** FUSE is brutally coarse: any single link failure between a node and the group collapses the entire group into a failure state. Applications have to decide whether converting link failures into group failures is acceptable — for short-lived leases or failure-of-lease-holder detection, it often is.

## Trade-offs and Alternatives

| Approach | Accuracy | Bandwidth | Partition Tolerance | Tuning Complexity | Notes |
|---|---|---|---|---|---|
| Ping / deadline | Moderate — depends on timeout | Low (O(N) per probing node) | Poor — a single bad link triggers false positive | Pick ping interval + timeout | Akka deadline detector; simplest baseline. |
| Timeout-free (counter vectors) | High — uses multi-path evidence | Medium — payload grows with path length | Strong — fair-path assumption absorbs link failures | Hard — no obvious threshold for counter gap | Fully asynchronous; academic rather than mainstream. |
| SWIM (outsourced heartbeats) | High — indirect probes cut false positives | Low/Medium — O(N) + k per miss | Strong — k vantage points | Moderate — k, T, subgroup size | Serf, Consul, Memberlist. |
| Phi-accrual | Very high in jittery networks — adapts | Low — same heartbeat stream as ping | Moderate — still a single-vantage view by default | Window W + phi threshold; easier in practice | Cassandra, Akka, Riak. |
| Gossip table | High aggregate — multi-vantage merge | Medium — O(N) msgs, O(N) payload | Strong — any path suffices | Gossip period + staleness timeout | Amazon-style membership; eventually-consistent view. |
| FUSE (quiescence) | Coarse (group-level only) | Very low — no broadcast needed | Very strong — silence crosses partitions | Minimal — define the group | Converts any failure into group failure; great for leases. |

**How to pick one.** Small cluster, LAN, simple semantics -> ping + deadline. Large cluster, unknown network -> SWIM or phi. WAN with partitions or a membership layer -> gossip. Lease / quorum / "all-or-nothing" group semantics -> FUSE.

## Key Takeaways

1. In an asynchronous system, perfect failure detection is impossible — every practical detector trades **accuracy against efficiency** and must tolerate some false positives.
2. The three-stage pipeline **Monitoring -> Interpretation -> Action** is the common shape; algorithms differ only in how each stage is implemented.
3. **Completeness, efficiency, and accuracy** are the three properties you use to compare detectors; [CHANDRA96] showed consensus is solvable even with a detector that makes infinitely many mistakes.
4. **Indirect evidence beats direct evidence**: SWIM's outsourced probes and gossip's merged tables both suppress false positives by consulting multiple vantage points.
5. **Phi-accrual replaces a hard timeout with a continuous score**, letting downstream code make graded decisions and letting the detector adapt to network jitter automatically.
6. **Quiescence is a propagation mechanism** — FUSE exploits the fact that an absent message carries information, at the cost of converting any single failure into a group failure.
7. Failure detection is the **liveness backbone** of everything that follows in the book — consensus, atomic broadcast, leader election, and replication all compose on top of a detector you can reason about.

## See Also

- [[01-failure-detector-fundamentals]]
- [[02-timeout-free-failure-detector]]
- [[03-swim-outsourced-heartbeats]]
- [[04-phi-accrual-failure-detector]]
- [[05-gossip-failure-detection]]
- [[06-fuse-reversing-failure-detection]]